## Engenharia de Variaveis

Constroi o dataset final para os modelos de deteccao de anomalias (Isolation Forest, LOF, One-Class SVM).

Entradas:
- `data/raw/propostas_pe.csv` — propostas por item de licitacao
- `data/raw/fornecedores_cnpj.csv` — dados cadastrais dos fornecedores
- `data/raw/municipios_pe.csv` — coordenadas dos municipios
- `data/raw/licitacoes_pe.csv` — metadados das contratacoes

Variaveis construidas:
1. **Preco**: desvio do valor proposto em relacao a mediana por categoria/item
2. **Concorrencia**: numero de propostas no mesmo item (fornecedor unico = suspeito)
3. **Temporal**: mes, ano, dia da semana da abertura
4. **Eleitoral**: distancia em dias para a eleicao municipal mais proxima (2020, 2024)
5. **Geografica**: distancia entre municipio do fornecedor e municipio do orgao
6. **Empresa**: idade da empresa na data da proposta, porte, coincidencia UF
7. **Historico**: frequencia do fornecedor em licitacoes do mesmo orgao

Saida: `data/processed/dataset_ml.csv`

In [1]:
import pandas as pd
import numpy as np
import json
import os
from datetime import datetime, date

os.makedirs("data/processed", exist_ok=True)
print("Setup OK")

Setup OK


### 1. Carrega e Inspeciona os Dados Brutos

In [2]:
propostas  = pd.read_csv("data/raw/propostas_pe.csv",          dtype=str)
licitacoes = pd.read_csv("data/raw/contratacoes14133_pe.csv",  dtype=str)
cnpj_df    = pd.read_csv("data/raw/fornecedores_cnpj.csv",     dtype=str)
municipios = pd.read_csv("data/raw/municipios_pe.csv",         dtype=str)

print(f"Propostas:   {len(propostas)} linhas | colunas: {list(propostas.columns)}")
print(f"Licitações:  {len(licitacoes)} linhas | colunas: {list(licitacoes.columns[:8])}...")
print(f"CNPJ:        {len(cnpj_df)} linhas")
print(f"Municípios:  {len(municipios)} linhas")

Propostas:   75502 linhas | colunas: ['idCompra', 'orgaoEntidadeCnpj', 'anoCompraPncp', 'sequencialCompraPncp', 'numeroItem', 'descricaoItem', 'cnpj_fornecedor', 'nome_fornecedor', 'valor_total_homologado', 'situacao_resultado']
Licitações:  11353 linhas | colunas: ['idCompra', 'numeroControlePNCP', 'anoCompraPncp', 'sequencialCompraPncp', 'orgaoEntidadeCnpj', 'orgaoSubrogadoCnpj', 'codigoOrgao', 'orgaoEntidadeRazaoSocial']...
CNPJ:        9458 linhas
Municípios:  185 linhas


### 2. Join: Propostas + Metadados da Licitacao

In [ ]:
JOIN_KEYS = ["orgaoEntidadeCnpj", "anoCompraPncp", "sequencialCompraPncp"]

for k in JOIN_KEYS:
    propostas[k]  = propostas[k].astype(str).str.strip()
    licitacoes[k] = licitacoes[k].astype(str).str.strip()

COLS_LIT = [
    *JOIN_KEYS,
    "unidadeOrgaoCodigoIbge",   # código IBGE do município do órgão
    "unidadeOrgaoMunicipioNome",# nome do município
    "modalidadeNome",
    "valorTotalEstimado",
    "dataPublicacaoPncp",
    "dataAberturaPropostaPncp",
]
cols_lit = [c for c in COLS_LIT if c in licitacoes.columns]

df = propostas.merge(
    licitacoes[cols_lit].drop_duplicates(subset=JOIN_KEYS),
    on=JOIN_KEYS,
    how="left"
)
print(f"Após join com licitações: {len(df)} linhas")
print(f"Colunas: {list(df.columns)}")

Após join com licitações: 75502 linhas
Colunas: ['idCompra', 'orgaoEntidadeCnpj', 'anoCompraPncp', 'sequencialCompraPncp', 'numeroItem', 'descricaoItem', 'cnpj_fornecedor', 'nome_fornecedor', 'valor_total_homologado', 'situacao_resultado', 'unidadeOrgaoCodigoIbge', 'unidadeOrgaoMunicipioNome', 'modalidadeNome', 'valorTotalEstimado', 'dataPublicacaoPncp', 'dataAberturaPropostaPncp']


### 3. Join: + Dados CNPJ do Fornecedor

In [ ]:
cnpj_forn_col = "cnpj_fornecedor"

cnpj_df["cnpj_norm"] = cnpj_df["cnpj"].astype(str).str.replace(r"\D", "", regex=True).str.zfill(14)
df["cnpj_forn_norm"] = df[cnpj_forn_col].astype(str).str.replace(r"\D", "", regex=True).str.zfill(14)

df["tipo_fornecedor"] = df["cnpj_forn_norm"].str.startswith("000").map(
    {True: "pessoa_fisica", False: "pessoa_juridica"}
)
print("Tipo de fornecedor:")
print(df["tipo_fornecedor"].value_counts())

COLS_CNPJ = [
    "cnpj_norm",
    "data_inicio_atividade",
    "porte",                      
    "natureza_juridica",         
    "municipio", "uf",
    "codigo_municipio_ibge",
    "cnae_fiscal",
    "capital_social",
    "descricao_situacao_cadastral",
    "opcao_pelo_mei",
    "opcao_pelo_simples",
]
cols_cnpj = [c for c in COLS_CNPJ if c in cnpj_df.columns]

df = df.merge(
    cnpj_df[cols_cnpj].rename(columns={"cnpj_norm": "cnpj_forn_norm"}),
    on="cnpj_forn_norm",
    how="left"
)
print(f"\nApós join com CNPJ: {len(df)} linhas")
print(f"porte nulos: {df['porte'].isna().sum() if 'porte' in df.columns else 'N/A'}")

Tipo de fornecedor:
tipo_fornecedor
pessoa_juridica    69446
pessoa_fisica       6056
Name: count, dtype: int64

Após join com CNPJ: 76218 linhas
porte nulos: 18740


### 4. Join: + Coordenadas IBGE

In [ ]:
municipios["codigoIbge"] = municipios["codigoIbge"].astype(str).str.strip()
municipios["latitude"]   = pd.to_numeric(municipios["latitude"],  errors="coerce")
municipios["longitude"]  = pd.to_numeric(municipios["longitude"], errors="coerce")

ibge_col = "unidadeOrgaoCodigoIbge"
if ibge_col in df.columns:
    df[ibge_col] = df[ibge_col].astype(str).str.strip()
    df = df.merge(
        municipios[["codigoIbge", "latitude", "longitude"]].rename(
            columns={"codigoIbge": ibge_col, "latitude": "lat_orgao", "longitude": "lon_orgao"}
        ),
        on=ibge_col,
        how="left"
    )

if "municipio" in df.columns:
    mun_coords = municipios[["nomeMunicipio", "latitude", "longitude"]].copy()
    mun_coords["nomeMunicipio"] = mun_coords["nomeMunicipio"].str.upper().str.strip()
    df["municipio_upper"] = df["municipio"].astype(str).str.upper().str.strip()
    df = df.merge(
        mun_coords.rename(columns={"nomeMunicipio": "municipio_upper", "latitude": "lat_forn", "longitude": "lon_forn"}),
        on="municipio_upper",
        how="left"
    )

print(f"Após joins de coordenadas: {len(df)} linhas")
print(f"lat_orgao nulos: {df['lat_orgao'].isna().sum() if 'lat_orgao' in df.columns else 'N/A'}")

Após joins de coordenadas: 76218 linhas
lat_orgao nulos: 0


### 5. Engenharia de Variaveis

In [ ]:
preco_col = "valor_total_homologado"
df[preco_col] = pd.to_numeric(df[preco_col], errors="coerce")

mediana_item = df.groupby(
    ["orgaoEntidadeCnpj", "sequencialCompraPncp", "numeroItem"]
)[preco_col].transform("median")
df["desvio_preco_mediana"] = (df[preco_col] - mediana_item) / mediana_item.replace(0, np.nan)

if "valorTotalEstimado" in df.columns:
    df["valorTotalEstimado"] = pd.to_numeric(df["valorTotalEstimado"], errors="coerce")
    df["ratio_preco_estimado"] = df[preco_col] / df["valorTotalEstimado"].replace(0, np.nan)
else:
    df["ratio_preco_estimado"] = np.nan

print(f"desvio_preco_mediana — não nulos: {df['desvio_preco_mediana'].notna().sum()}")
print(f"ratio_preco_estimado — não nulos: {df['ratio_preco_estimado'].notna().sum()}")

desvio_preco_mediana — não nulos: 75578
ratio_preco_estimado — não nulos: 75388


In [ ]:
# --- 5.2 Concorrência ---
n_propostas = df.groupby(
    ["orgaoEntidadeCnpj", "sequencialCompraPncp", "numeroItem"]
)["cnpj_forn_norm"].transform("count")

df["n_propostas_item"] = n_propostas
df["fornecedor_unico"] = (n_propostas == 1).astype(int)
print(f"Items com fornecedor único: {df['fornecedor_unico'].sum()} ({df['fornecedor_unico'].mean()*100:.1f}%)")

Items com fornecedor único: 68397 (89.7%)


In [8]:
# --- 5.3 Variaveis temporais ---
data_col = "dataAberturaProposta" if "dataAberturaProposta" in df.columns else "dataPublicacaoPncp"
df["data_abertura"] = pd.to_datetime(df[data_col], errors="coerce")

df["ano"] = df["data_abertura"].dt.year
df["mes"] = df["data_abertura"].dt.month
df["dia_semana"] = df["data_abertura"].dt.dayofweek  # 0=segunda, 6=domingo
df["trimestre"] = df["data_abertura"].dt.quarter
print("Variaveis temporais criadas")

Variaveis temporais criadas


In [9]:
# --- 5.4 Variavel eleitoral ---
# Eleicoes municipais: 15/11/2020, 01/10/2024
ELEICOES = [date(2020, 11, 15), date(2024, 10, 6)]

def dist_eleicao(d):
    if pd.isna(d):
        return np.nan
    d_date = d.date()
    return min(abs((d_date - e).days) for e in ELEICOES)

df["dias_para_eleicao"] = df["data_abertura"].apply(dist_eleicao)
df["ano_eleitoral"] = df["ano"].isin([2020, 2024]).astype(int)
df["pre_eleitoral_180d"] = (df["dias_para_eleicao"] <= 180).astype(int)
print(f"Propostas em periodo pre-eleitoral (180d): {df['pre_eleitoral_180d'].sum()}")

Propostas em periodo pre-eleitoral (180d): 45045


In [10]:
# --- 5.5 Distancia geografica fornecedor x orgao ---
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

if all(c in df.columns for c in ["lat_orgao", "lon_orgao", "lat_forn", "lon_forn"]):
    df["dist_km_forn_orgao"] = haversine(
        df["lat_forn"].astype(float),
        df["lon_forn"].astype(float),
        df["lat_orgao"].astype(float),
        df["lon_orgao"].astype(float)
    )
    df["fornecedor_fora_pe"] = (~df["uf"].fillna("").str.upper().eq("PE")).astype(int)
    print(f"Distancia media forn-orgao: {df['dist_km_forn_orgao'].mean():.1f} km")
else:
    print("Aviso: colunas de coordenadas nao encontradas, variavel de distancia nao criada")

Distancia media forn-orgao: 61.7 km


In [11]:
# --- 5.6 Variáveis da empresa ---
if "data_inicio_atividade" in df.columns:
    df["data_abertura_empresa"] = pd.to_datetime(df["data_inicio_atividade"], errors="coerce")
    df["idade_empresa_anos"] = (
        (df["data_abertura"] - df["data_abertura_empresa"]).dt.days / 365.25
    ).clip(lower=0)
    df["empresa_nova"] = (df["idade_empresa_anos"] < 1).astype(int)
    print(f"Idade média das empresas: {df['idade_empresa_anos'].mean():.1f} anos")

# Frequência do fornecedor em licitações do órgão
# (quantas contratações distintas o fornecedor ganhou neste órgão)
freq_forn_orgao = df.groupby(
    ["orgaoEntidadeCnpj", "cnpj_forn_norm"]
)["sequencialCompraPncp"].transform("nunique")
df["freq_forn_orgao"] = freq_forn_orgao

print("Variáveis de empresa criadas")

Idade média das empresas: 12.2 anos
Variáveis de empresa criadas


### 6. Dataset Final

In [12]:
# Seleciona colunas finais para o modelo
FEATURES_ML = [
    # Identificadores
    "orgaoEntidadeCnpj", "anoCompraPncp", "sequencialCompraPncp", "numeroItem", "cnpj_forn_norm",
    # Preço
    "desvio_preco_mediana", "ratio_preco_estimado",
    # Concorrência
    "n_propostas_item", "fornecedor_unico",
    # Temporal
    "ano", "mes", "trimestre", "dia_semana",
    # Eleitoral
    "dias_para_eleicao", "ano_eleitoral", "pre_eleitoral_180d",
    # Geográfica
    "dist_km_forn_orgao", "fornecedor_fora_pe",
    # Empresa
    "tipo_fornecedor", "idade_empresa_anos", "empresa_nova", "freq_forn_orgao",
    "opcao_pelo_mei", "opcao_pelo_simples",
    # Contexto (categóricas — codificar antes do modelo)
    "modalidadeNome", "porte", "cnae_fiscal",
]

cols_presentes = [c for c in FEATURES_ML if c in df.columns]
cols_ausentes  = [c for c in FEATURES_ML if c not in df.columns]

print(f"Features presentes ({len(cols_presentes)}): {cols_presentes}")
if cols_ausentes:
    print(f"Ausentes: {cols_ausentes}")

dataset_ml = df[cols_presentes].copy()
dataset_ml.to_csv("data/processed/dataset_ml.csv", index=False)
print(f"\nDataset salvo: {len(dataset_ml)} linhas x {len(dataset_ml.columns)} colunas")
print("\nEstatísticas:")
dataset_ml.describe()

Features presentes (27): ['orgaoEntidadeCnpj', 'anoCompraPncp', 'sequencialCompraPncp', 'numeroItem', 'cnpj_forn_norm', 'desvio_preco_mediana', 'ratio_preco_estimado', 'n_propostas_item', 'fornecedor_unico', 'ano', 'mes', 'trimestre', 'dia_semana', 'dias_para_eleicao', 'ano_eleitoral', 'pre_eleitoral_180d', 'dist_km_forn_orgao', 'fornecedor_fora_pe', 'tipo_fornecedor', 'idade_empresa_anos', 'empresa_nova', 'freq_forn_orgao', 'opcao_pelo_mei', 'opcao_pelo_simples', 'modalidadeNome', 'porte', 'cnae_fiscal']

Dataset salvo: 76218 linhas x 27 colunas

Estatísticas:


,desvio_preco_mediana,ratio_preco_estimado,n_propostas_item,fornecedor_unico,ano,mes,trimestre,dia_semana,dias_para_eleicao,ano_eleitoral,pre_eleitoral_180d,dist_km_forn_orgao,fornecedor_fora_pe,idade_empresa_anos,empresa_nova,freq_forn_orgao
count,75578.000000,75388.000000,76218.000000,76218.000000,76218.000000,76218.000000,76218.000000,76218.000000,76218.000000,76218.000000,76218.000000,24588.000000,76218.000000,57478.000000,76218.000000,76218.000000
mean,0.614842,0.122653,1.485581,0.897386,2023.723307,7.415046,2.795731,2.036278,186.154137,0.723307,0.591002,61.685767,0.631150,12.239447,0.068435,3.226482
std,159.971668,0.779655,2.982230,0.303455,0.447366,3.121319,1.005885,1.431250,155.353153,0.447366,0.491652,133.449606,0.482496,11.838586,0.252493,4.679032
min,-1.000000,0.000000,1.000000,0.000000,2023.000000,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
25%,0.000000,0.001351,1.000000,1.000000,2023.000000,5.000000,2.000000,1.000000,57.000000,0.000000,0.000000,0.000000,0.000000,3.151266,0.000000,1.000000
50%,0.000000,0.005773,1.000000,1.000000,2024.000000,8.000000,3.000000,2.000000,138.000000,1.000000,1.000000,0.000000,1.000000,8.659822,0.000000,2.000000
75%,0.000000,0.040551,1.000000,1.000000,2024.000000,10.000000,4.000000,3.000000,305.000000,1.000000,1.000000,55.918158,1.000000,17.349760,0.000000,3.000000
max,43966.479675,200.000000,42.000000,1.000000,2024.000000,12.000000,4.000000,6.000000,641.000000,1.000000,1.000000,640.924871,1.000000,116.331280,1.000000,46.000000


In [ ]:
missing = dataset_ml.isnull().sum()
missing_pct = (missing / len(dataset_ml) * 100).round(1)
relatorio = pd.DataFrame({"ausentes": missing, "pct": missing_pct})
print("Valores ausentes por coluna:")
print(relatorio[relatorio["ausentes"] > 0].sort_values("pct", ascending=False))

Valores ausentes por coluna:
                      ausentes   pct
dist_km_forn_orgao       51630  67.7
opcao_pelo_mei           29803  39.1
opcao_pelo_simples       29803  39.1
idade_empresa_anos       18740  24.6
porte                    18740  24.6
cnae_fiscal              18740  24.6
ratio_preco_estimado       830   1.1
desvio_preco_mediana       640   0.8
